# Format H2 handwriting dataset using Pytorch

New version: save average and std, using day-wise normalization  
Meta training split

- We need to remove the first 2 sentences from calibration dataset

Reference: 
- [original data description](https://datadryad.org/stash/dataset/doi:10.5061/dryad.hqbzkh1p6#readme)
- [H2 dataset description](https://dandiarchive.org/dandiset/000950)
- [formating codes for speechBCI](https://github.com/cffan/neural_seq_decoder/blob/master/notebooks/formatCompetitionData.ipynb)

In [1]:
from pathlib import Path
import numpy as np
from pynwb import NWBHDF5IO
import warnings
warnings.filterwarnings("ignore")

## Load data

- meta train
    - training & test: day 1-18, calibration set
    - validation: day 1-18, minival set
- meta test
    - few-shot adaptation: day 19-21, calibration set, first k samples
    - few-shot test: day 19-21, minival set

In [2]:
data_dir = Path("../data/000950/")
if not data_dir.exists():
    raise FileNotFoundError("Data not found. Please make sure to run data setup first (e.g. ./download_falcon_datasets.sh).")

train_path = Path(data_dir / 'sub-T5-held-in-calib')
test_path = Path(data_dir / 'sub-T5-held-in-minival')

meta_train_files = sorted(train_path.glob('*.nwb'))[:18]
meta_val_files = sorted(test_path.glob('*.nwb'))[:18]
meta_fs_files = sorted(train_path.glob('*.nwb'))[18:]
meta_test_files = sorted(test_path.glob('*.nwb'))[18:]

In each datafile, the `acquisitions` field contains binned_spikes (binned at 20ms) and the `eval_mask` used for determining valid FALCON evaluation periods. The `trials` field contains trial metadata including the prompt that T5 was asked to attempt to write (`cue`) and the experimental block that each trial belongs to (`block_num`).

In [3]:
def load_nwb(fn: str):
    print(f'Loading {fn}')
    with NWBHDF5IO(fn, 'r') as io:
        nwbfile = io.read()

        trial_info = (
            nwbfile.trials.to_dataframe()
            .reset_index()
        )

        binned_spikes = nwbfile.acquisition['binned_spikes'].data[()]
        time = nwbfile.acquisition['binned_spikes'].timestamps[()]
        eval_mask = nwbfile.acquisition['eval_mask'].data[()].astype(bool)

    return(
        binned_spikes,
        time,
        trial_info,
        eval_mask
    )

## Record avg and std for each day's data

Use calibration dataset

In [4]:
# hashed by sessions
all_avg = {}
all_std = {}

In [5]:
def hash_dataset(handle):
        r"""
            handle - path.stem of a datafile.
            Convenience function to help identify what "session" a datafile belongs to.. If multiple files per session in real-world time, this may _not_ uniquely identify runfile.
        """
        if isinstance(handle, Path):
            handle = handle.stem
        else:
              raise TypeError("input should be a datafile!")
        return handle.split('_')[1]

In [6]:
def h2_relabel_hotfix(dandi_tag):
    if dandi_tag.startswith('ses'):
        evaluator_tag = dandi_tag.split('-')[1]
        evaluator_tag = evaluator_tag[:4] + '.' + evaluator_tag[4:6] + '.' + evaluator_tag[6:]
        return evaluator_tag
    return dandi_tag

In [7]:
for fn in meta_train_files:
    spikes, time, tinfo, eval_mask = load_nwb(fn)
    dataset_tag = hash_dataset(fn)
    dataset_tag = h2_relabel_hotfix(dataset_tag)
    feats_mean = np.mean(spikes, axis=0, keepdims=True)
    feats_std = np.std(spikes, axis=0, keepdims=True)
    all_avg[dataset_tag] = feats_mean
    all_std[dataset_tag] = feats_std

Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220518.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220523.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220525.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220601.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220603.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220606.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220608.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220613.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220615.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220622.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220901.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220929.nwb
Loading ../data/

In [8]:
for fn in meta_fs_files:
    spikes, time, tinfo, eval_mask = load_nwb(fn)
    dataset_tag = hash_dataset(fn)
    dataset_tag = h2_relabel_hotfix(dataset_tag)
    feats_mean = np.mean(spikes, axis=0, keepdims=True)
    feats_std = np.std(spikes, axis=0, keepdims=True)
    all_avg[dataset_tag] = feats_mean
    all_std[dataset_tag] = feats_std

Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20221208.nwb


Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20221215.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20230228.nwb


In [9]:
# import pickle
# avg_and_std = {}
# avg_and_std['avg'] = all_avg
# avg_and_std['std'] = all_std
# with open('avg_std_remote', 'wb') as handle:
#     pickle.dump(avg_and_std, handle)

In [10]:
all_avg.keys()

dict_keys(['2022.05.18', '2022.05.23', '2022.05.25', '2022.06.01', '2022.06.03', '2022.06.06', '2022.06.08', '2022.06.13', '2022.06.15', '2022.06.22', '2022.09.01', '2022.09.29', '2022.10.06', '2022.10.18', '2022.10.25', '2022.10.27', '2022.11.01', '2022.11.03', '2022.12.08', '2022.12.15', '2023.02.28'])

## Read features from the data

Spikes are recorded into one single array, we need to find the spike for each trial.

In [11]:
def loadFeatures(sessionPath):
    spikes, time, tinfo, eval_mask = load_nwb(sessionPath)
    dataset_tag = hash_dataset(fn)
    dataset_tag = h2_relabel_hotfix(dataset_tag)

    input_features = []
    transcriptions = []
    frame_lens = []

    local_avg = all_avg[dataset_tag]
    local_std = all_std[dataset_tag]

    for _, row in tinfo.iterrows():
    # find the spikes for each trial
        trail_start = row['start_time']
        trail_end = row['stop_time']
        start_id = np.where(time==trail_start)[0][0]
        end_id = np.where(time==trail_end)[0][0]
        features = spikes[start_id:end_id+1]
        sentence_len = features.shape[0]
        sentence = row['cue']

        # normalization
        features = (features - local_avg) / (local_std + 1e-8)

        input_features.append(features)
        transcriptions.append(sentence)
        frame_lens.append(sentence_len)

    #convert to tfRecord file
    session_data = {
        'inputFeatures': input_features,
        'transcriptions': transcriptions,
        'frameLens': frame_lens
    }

    return session_data

## Data preprocessing

- Use char id

Charactre dictionary

In [12]:
CHAR_DEF = ['a', 'b', 'c', 'd', 'e', 'f', 'g',
            'h', 'i', 'j', 'k', 'l', 'm', 'n',
            'o', 'p', 'q', 'r', 's', 't',
            'u', 'v', 'w', 'x', 'y', 'z',
            '>', ',', "'", '~', '?']

def charToId(c):
    return CHAR_DEF.index(c)

In [13]:
def getDataset(fileName, calibration=True):
    session_data = loadFeatures(fileName)
        
    allDat = []
    trueSentences = []
    seqElements = []

    if calibration: # first 2 sentences are the same as minival dataset
        trail_range = range(2, len(session_data['inputFeatures']))
    else:
        trail_range = range(len(session_data['inputFeatures']))

    # for each trial
    for x in trail_range:
        allDat.append(session_data['inputFeatures'][x])
        trueSen = session_data['transcriptions'][x]
        trueSentences.append(trueSen)

        maxSeqLen = 200 # max char len is 117
        seqClassIDs = np.zeros([maxSeqLen]).astype(np.int32)
        seqLen = len(trueSen)
        seqClassIDs[0:seqLen] = [charToId(c) + 1 for c in trueSen]
        seqElements.append(seqClassIDs)

    newDataset = {}
    newDataset['sentenceDat'] = allDat
    newDataset['transcriptions'] = trueSentences
    newDataset['chars'] = seqElements

    charLens = []
    for x in range(len(newDataset['sentenceDat'])):
        zeroIdx = np.argwhere(newDataset['chars'][x]==0)
        charLens.append(zeroIdx[0,0])
    
    newDataset['charLens'] = np.array(charLens)
    
    return newDataset

In [14]:
def getHeldout(fileName):
    session_data = loadFeatures(fileName) 
    total_trials = len(session_data['inputFeatures'])
    
    def prepare_dataset(trial_indices):
        allDat = []
        trueSentences = []
        seqElements = []
        
        for idx in trial_indices:
            allDat.append(session_data['inputFeatures'][idx])
            trueSen = session_data['transcriptions'][idx]
            trueSentences.append(trueSen)
            
            maxSeqLen = 200
            seqClassIDs = np.zeros([maxSeqLen]).astype(np.int32)
            seqLen = len(trueSen)
            seqClassIDs[0:seqLen] = [charToId(c) + 1 for c in trueSen]
            seqElements.append(seqClassIDs)
        
        charLens = []
        for seq in seqElements:
            zeroIdx = np.where(seq == 0)[0]
            charLens.append(zeroIdx[0])
        
        return {
            'sentenceDat': allDat,
            'transcriptions': trueSentences,
            'chars': seqElements,
            'charLens': np.array(charLens)
        }
    
    test_size = 5
    calib_indices = list(range(0, total_trials - test_size))
    test_indices = list(range(total_trials - test_size, total_trials))
    
    calibrationDataset = prepare_dataset(calib_indices)
    testDataset = prepare_dataset(test_indices)
    
    return calibrationDataset, testDataset

## Save the data

In [15]:
trainDatasets = []
valDatasets = []
fsDatasets = []
testDatasets = []

In [16]:
# for fn in meta_train_files:
#     trainDataset = getDataset(fn, calibration=True)
#     trainDatasets.append(trainDataset)

In [17]:
# for fn in meta_val_files:
#     valDataset = getDataset(fn, calibration=False)
#     valDatasets.append(valDataset)

In [18]:
# for fn in meta_fs_files:
#     fsDataset = getDataset(fn, calibration=True)
#     fsDatasets.append(fsDataset)

In [19]:
# for fn in meta_test_files:
#     testDataset = getDataset(fn, calibration=False)
#     testDatasets.append(testDataset)

In [20]:
# # Check if there is overlapping in the training dataset and validation dataset
# assert len(trainDatasets)==len(valDatasets)
# for i in range(len(trainDatasets)):
#     print("="*50)
#     print(f"Day ID: {i}")
#     print(f"Number of the calibration trials: {len(trainDatasets[i]['sentenceDat'])}")
#     print(f"Number of the minival trials: {len(valDatasets[i]['sentenceDat'])}")
#     has_intersection = not set(trainDatasets[i]['transcriptions']).isdisjoint(valDatasets[i]['transcriptions'])
#     print("Overlapping exist") if has_intersection else print("No overlapping") 

In [21]:
# # Check if there is overlapping in the few-shot dataset and test dataset
# assert len(fsDatasets)==len(testDatasets)
# for i in range(len(fsDatasets)):
#     print("="*50)
#     print(f"Day ID: {len(trainDatasets)+i}")
#     print(f"Number of the calibration trials: {len(fsDatasets[i]['sentenceDat'])}")
#     print(f"Number of the minival trials: {len(testDatasets[i]['sentenceDat'])}")
#     has_intersection = not set(fsDatasets[i]['transcriptions']).isdisjoint(testDatasets[i]['transcriptions'])
#     print("Overlapping exist") if has_intersection else print("No overlapping") 

In [22]:
heldout_calibs = []
heldout_tests = []
for fn in meta_fs_files:
    heldout_calib, heldout_test = getHeldout(fn)
    heldout_calibs.append(heldout_calib)
    heldout_tests.append(heldout_test)

Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20221208.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20221215.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20230228.nwb


In [23]:
# Check if there is overlapping in the few-shot dataset and test dataset
assert len(heldout_calibs)==len(heldout_tests)
for i in range(len(heldout_calibs)):
    print("="*50)
    print(f"Day ID: {18+i}")
    print(f"Number of the calibration trials: {len(heldout_calibs[i]['sentenceDat'])}")
    print(f"Number of the minival trials: {len(heldout_tests[i]['sentenceDat'])}")
    has_intersection = not set(heldout_calibs[i]['transcriptions']).isdisjoint(heldout_tests[i]['transcriptions'])
    print("Overlapping exist") if has_intersection else print("No overlapping") 

Day ID: 18
Number of the calibration trials: 19
Number of the minival trials: 5
No overlapping
Day ID: 19
Number of the calibration trials: 7
Number of the minival trials: 5
No overlapping
Day ID: 20
Number of the calibration trials: 19
Number of the minival trials: 5
No overlapping


In [ ]:
import pickle

Heldout = {}
Heldout['fs'] = heldout_calibs
Heldout['test'] = heldout_tests

with open('local_heldout', 'wb') as handle:
    pickle.dump(Heldout, handle)

In [25]:
# import pickle

# allDatasets = {}
# allDatasets['train'] = trainDatasets
# allDatasets['val'] = valDatasets
# allDatasets['fs'] = fsDatasets
# allDatasets['test'] = testDatasets

# with open('data_local_no_overlap', 'wb') as handle:
#     pickle.dump(allDatasets, handle)